# Challenge Packaging Industries — Data Cleaning
**Analyst:** Divya Bhuvaneshwaran  
**Data:** Day Book FY 2024-25 + FY 2025-26  
**Output:** `day_book_final.csv` — clean, combined, deduplicated  

## What this notebook does:
1. Reads raw Day Book Excel files for both financial years
2. Cleans and standardises column names and data types
3. Adds time intelligence columns (FY, Month, Quarter, Day)
4. Categorises each transaction (Sales, Purchase, Receipt, Payment etc.)
5. Removes duplicates from Jan-Mar 2026 overlap period
6. Saves final clean combined CSV ready for Power BI

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported!')
print(f'   pandas version: {pd.__version__}')

## Step 2 — File Paths

In [ ]:
# ── Update these paths to match your local setup ──
BASE_PATH = r'C:\Users\divya\Documents\challenge-packaging-analysis\data'

FILE_2425 = os.path.join(BASE_PATH, 'DayBook_24-25_DIV__1_.xlsx')
FILE_2526 = os.path.join(BASE_PATH, '25-26_Daybook.xls')
OUTPUT_FILE = os.path.join(BASE_PATH, 'day_book_final.csv')

print('File paths configured:')
print(f'  24-25 file: {FILE_2425}')
print(f'  25-26 file: {FILE_2526}')
print(f'  Output:     {OUTPUT_FILE}')
print()

# Check files exist
for path, name in [(FILE_2425, '24-25'), (FILE_2526, '25-26')]:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f'  ✅ {name} file found ({size:.1f} KB)')
    else:
        print(f'  ❌ {name} file NOT found — check path!')

## Step 3 — Transaction Categorisation Rules

In [ ]:
def categorise_transaction(vch_type, particulars):
    """
    Categorises each transaction based on Voucher Type.
    
    Categories:
    - Sales          : Revenue from carton box sales (Debit side)
    - Purchase       : Raw material purchases (Credit side)
    - Purchase Order : Customer order confirmations — NOT actual purchases
    - Receipt        : Actual cash received from customers (Credit side)
    - Payment        : Actual cash paid out — suppliers, loans, salaries (Debit side)
    - Journal        : Operating expenses — electricity, rent, petty cash (Debit side)
    - Contra         : Internal bank transfers — NOT real cash movement
    - Credit Note    : Sales returns/refunds — reduces total sales
    - Debit Note     : Additional charges to customers — adds to sales
    - Delivery Challan: Delivery documents — no financial impact
    - Other          : Anything else
    
    Note: Journal entries include electricity, rent, petty cash — confirmed
    by business owner Bhuvanesh. These are real operating expenses.
    
    Note: Contra entries are internal bank-to-bank transfers (e.g. Union Bank
    to Indian Bank for OD payments) — NOT included in Cash Out.
    """
    vt = str(vch_type).strip()
    
    if vt in ['GST Sales New', 'GST SALES NEW 25-26']:
        return 'Sales'
    elif vt in ['Purchase', 'Purchase New']:
        return 'Purchase'
    elif vt == 'Purchase Order':
        return 'Purchase Order'  # customer order — not actual purchase
    elif vt in ['Receipt', 'Receipt New', 'Receipt 25-26']:
        return 'Receipt'
    elif vt in ['Payment', 'Payment New']:
        return 'Payment'
    elif vt == 'Journal New':
        return 'Journal'
    elif vt == 'Contra':
        return 'Contra'  # internal transfer — excluded from cash flow
    elif vt == 'Credit Note New':
        return 'Credit Note'
    elif vt == 'Debit Note New':
        return 'Debit Note'
    elif vt in ['Delivery Challan', 'Delivery Challan New']:
        return 'Delivery Challan'
    else:
        return 'Other'

print('✅ Categorisation function defined!')
print()
print('Categories and their business meaning:')
categories = [
    ('Sales',           'Revenue — Debit_Amount is the sale value'),
    ('Purchase',        'Raw material cost — Credit_Amount is the purchase value'),
    ('Purchase Order',  'Customer order confirmation — excluded from financials'),
    ('Receipt',         'Actual cash received — Credit_Amount'),
    ('Payment',         'Actual cash paid out — Debit_Amount (includes loans, salary)'),
    ('Journal',         'Operating expenses — Debit_Amount (electricity, rent, petty cash)'),
    ('Contra',          'Internal bank transfer — excluded from cash flow analysis'),
    ('Credit Note',     'Sales return/refund — Credit_Amount reduces total sales'),
    ('Debit Note',      'Extra charge to customer — Debit_Amount adds to total sales'),
    ('Delivery Challan','Delivery document — no financial impact'),
]
for cat, meaning in categories:
    print(f'  {cat:<20} → {meaning}')

## Step 4 — Data Reading and Cleaning Function

In [ ]:
def read_and_clean_daybook(filepath, fy_label):
    """
    Reads a raw Tally Day Book Excel file and returns a clean DataFrame.
    
    Parameters:
        filepath  : Path to the Excel file
        fy_label  : Financial year label e.g. '2024-25'
    
    Returns:
        Clean DataFrame with all required columns
    """
    print(f'Reading {fy_label} data...')
    
    # ── 1. Read raw file ──
    # Skip first 9 rows — company header, address, date range
    df = pd.read_excel(
        filepath,
        engine='openpyxl',
        sheet_name='Day Book',
        skiprows=9
    )
    
    # ── 2. Standardise column names ──
    df.columns = ['Date', 'Particulars', 'Vch_Type', 'Vch_No', 
                  'Debit_Amount', 'Credit_Amount']
    
    # ── 3. Remove header rows that Tally repeats mid-file ──
    df = df[df['Date'] != 'Date']
    df = df[df['Date'] != 'Inwards Qty']
    
    # ── 4. Parse date column ──
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.dropna(subset=['Date'])
    
    # ── 5. Parse numeric columns ──
    df['Debit_Amount'] = pd.to_numeric(df['Debit_Amount'], errors='coerce').fillna(0)
    df['Credit_Amount'] = pd.to_numeric(df['Credit_Amount'], errors='coerce').fillna(0)
    
    # ── 6. Clean text columns ──
    df['Particulars'] = df['Particulars'].astype(str).str.strip()
    df['Vch_Type'] = df['Vch_Type'].astype(str).str.strip()
    df['Vch_No'] = df['Vch_No'].astype(str).str.strip()
    
    # ── 7. Remove summary/total rows ──
    remove_keywords = ['nan', 'total', 'closing balance', 'opening balance', '']
    df = df[~df['Particulars'].str.lower().isin(remove_keywords)]
    df = df[~df['Particulars'].str.lower().str.startswith('total')]
    df = df[~df['Particulars'].str.lower().str.startswith('grand')]
    
    # ── 8. Add Financial Year label ──
    df['FY'] = fy_label
    
    # ── 9. Add time intelligence columns (Indian FY: Apr=Month 1) ──
    df['Month_Name'] = df['Date'].dt.strftime('%b')  # Jan, Feb, Mar...
    df['Day_of_Week'] = df['Date'].dt.strftime('%A') # Monday, Tuesday...
    
    # Indian FY month numbering: Apr=1, May=2 ... Mar=12
    month_to_num = {4:1, 5:2, 6:3, 7:4, 8:5, 9:6,
                    10:7, 11:8, 12:9, 1:10, 2:11, 3:12}
    df['Month_Num'] = df['Date'].dt.month.map(month_to_num)
    
    # Indian FY quarters: Q1=Apr-Jun, Q2=Jul-Sep, Q3=Oct-Dec, Q4=Jan-Mar
    month_to_q = {4:'Q1', 5:'Q1', 6:'Q1',
                  7:'Q2', 8:'Q2', 9:'Q2',
                  10:'Q3', 11:'Q3', 12:'Q3',
                  1:'Q4', 2:'Q4', 3:'Q4'}
    df['Quarter'] = df['Date'].dt.month.map(month_to_q)
    
    # ── 10. Categorise each transaction ──
    df['Txn_Category'] = df.apply(
        lambda r: categorise_transaction(r['Vch_Type'], r['Particulars']), 
        axis=1
    )
    
    # ── 11. Reset index ──
    df = df.reset_index(drop=True)
    
    print(f'  ✅ {fy_label}: {len(df):,} rows | {df.Date.min().date()} to {df.Date.max().date()}')
    return df

print('✅ Cleaning function defined!')

## Step 5 — Read Both Years

In [ ]:
# Read and clean both financial years
df_2425 = read_and_clean_daybook(FILE_2425, '2024-25')
df_2526 = read_and_clean_daybook(FILE_2526, '2025-26')

print()
print('Transaction category breakdown:')
print()
print('FY 2024-25:')
print(df_2425.groupby('Txn_Category').size().sort_values(ascending=False).to_string())
print()
print('FY 2025-26:')
print(df_2526.groupby('Txn_Category').size().sort_values(ascending=False).to_string())

## Step 6 — Remove Duplicates from Overlap Period

**Why duplicates exist:**  
The 24-25 Day Book file contains entries from Jan-Mar 2026 (last quarter).  
The 25-26 Day Book file also contains those same Jan-Mar 2026 entries.  
We keep the 25-26 file as the authoritative source for Jan-Mar 2026.

In [ ]:
# ── Identify overlap period ──
OVERLAP_START = pd.Timestamp('2026-01-01')

# Split 24-25 data into two parts:
# Part 1: Apr 2024 to Dec 2025 — no overlap, keep all
df_2425_pre_overlap = df_2425[df_2425['Date'] < OVERLAP_START]

# Part 2: Jan 2026 to Mar 2026 — potential duplicates
df_2425_overlap = df_2425[df_2425['Date'] >= OVERLAP_START]
df_2526_overlap = df_2526[df_2526['Date'] >= OVERLAP_START]

# Find which Vch_Nos in 24-25 overlap also exist in 25-26
vch_nos_in_2526 = set(df_2526_overlap['Vch_No'].dropna())
df_2425_unique_overlap = df_2425_overlap[
    ~df_2425_overlap['Vch_No'].isin(vch_nos_in_2526)
]

print('Overlap period analysis (Jan-Mar 2026):')
print(f'  24-25 file entries: {len(df_2425_overlap):,}')
print(f'  25-26 file entries: {len(df_2526_overlap):,}')
print(f'  Duplicates removed: {len(df_2425_overlap) - len(df_2425_unique_overlap):,}')
print(f'  Unique 24-25 entries kept: {len(df_2425_unique_overlap):,}')
print()
print('Strategy: Keep 25-26 file as authoritative source for Jan-Mar 2026')

## Step 7 — Combine and Save Final Dataset

In [ ]:
# ── Combine all parts ──
df_final = pd.concat([
    df_2425_pre_overlap,     # Apr 2024 - Dec 2025 from 24-25 file
    df_2425_unique_overlap,  # Any unique Jan-Mar 2026 from 24-25 file
    df_2526_overlap          # Jan-Mar 2026 from 25-26 file (authoritative)
], ignore_index=True)

# Sort by date
df_final = df_final.sort_values('Date').reset_index(drop=True)

print(f'Final dataset: {len(df_final):,} rows')
print(f'Date range: {df_final.Date.min().date()} to {df_final.Date.max().date()}')
print(f'Columns: {list(df_final.columns)}')
print()
print('Rows per FY:')
print(df_final.groupby('FY').size().to_string())

# ── Save ──
df_final.to_csv(OUTPUT_FILE, index=False)
print()
print(f'✅ Saved to: {OUTPUT_FILE}')

## Step 8 — Verify Financial Figures

In [ ]:
def verify_financials(df, fy):
    """
    Prints key financial metrics for a given FY.
    These should match your Power BI DAX measures.
    """
    d = df[df['FY'] == fy]
    
    # Sales components
    sales       = d[d['Txn_Category']=='Sales']['Debit_Amount'].sum()
    debit_note  = d[d['Txn_Category']=='Debit Note']['Debit_Amount'].sum()
    credit_note = d[d['Txn_Category']=='Credit Note']['Credit_Amount'].sum()
    net_sales   = sales + debit_note - credit_note
    
    # Cost
    purchase    = d[d['Txn_Category']=='Purchase']['Credit_Amount'].sum()
    
    # Profit
    gross_margin    = net_sales - purchase
    operating_exp   = d[d['Txn_Category']=='Journal']['Debit_Amount'].sum()
    net_profit      = gross_margin - operating_exp
    
    # Cash flow
    cash_in     = d[d['Txn_Category']=='Receipt']['Credit_Amount'].sum()
    cash_out    = d[d['Txn_Category'].isin(['Payment','Journal'])]['Debit_Amount'].sum()
    net_cf      = cash_in - cash_out
    
    # Owner drawings
    drawings    = d[
        d['Particulars'].str.lower().str.contains('capital', na=False) &
        (d['Txn_Category'] == 'Payment')
    ]['Debit_Amount'].sum()
    
    # Loan payments
    loans       = d[
        d['Particulars'].str.lower().str.contains('loan', na=False) &
        (d['Txn_Category'] == 'Payment')
    ]['Debit_Amount'].sum()
    
    print(f'=== FY {fy} ===')
    print(f'  REVENUE')
    print(f'    Base Sales:          {sales/10000000:.2f} Cr')
    print(f'    + Debit Notes:       {debit_note/10000000:.2f} Cr')
    print(f'    - Credit Notes:      {credit_note/10000000:.2f} Cr')
    print(f'    Net Sales:           {net_sales/10000000:.2f} Cr')
    print()
    print(f'  PROFITABILITY')
    print(f'    Purchase Cost:       {purchase/10000000:.2f} Cr')
    print(f'    Gross Margin:        {gross_margin/10000000:.2f} Cr ({gross_margin/net_sales*100:.1f}%)')
    print(f'    Operating Exp:       {operating_exp/10000000:.2f} Cr')
    print(f'    Net Profit:          {net_profit/10000000:.2f} Cr ({net_profit/net_sales*100:.1f}%)')
    print()
    print(f'  CASH FLOW')
    print(f'    Cash In (Receipts):  {cash_in/10000000:.2f} Cr')
    print(f'    Cash Out:            {cash_out/10000000:.2f} Cr')
    print(f'    Net Cash Flow:       {net_cf/10000000:.2f} Cr')
    print()
    print(f'  KEY ITEMS IN CASH OUT')
    print(f'    Owner Drawings:      {drawings/10000000:.2f} Cr')
    print(f'    Loan Payments:       {loans/10000000:.2f} Cr')
    print()

verify_financials(df_final, '2024-25')
verify_financials(df_final, '2025-26')

## Step 9 — Year-on-Year Comparison

In [ ]:
print('=== YEAR-ON-YEAR COMPARISON ===')
print()

metrics = {}
for fy in ['2024-25', '2025-26']:
    d = df_final[df_final['FY']==fy]
    sales       = d[d['Txn_Category']=='Sales']['Debit_Amount'].sum()
    debit_note  = d[d['Txn_Category']=='Debit Note']['Debit_Amount'].sum()
    credit_note = d[d['Txn_Category']=='Credit Note']['Credit_Amount'].sum()
    net_sales   = sales + debit_note - credit_note
    purchase    = d[d['Txn_Category']=='Purchase']['Credit_Amount'].sum()
    gross_margin = net_sales - purchase
    operating_exp = d[d['Txn_Category']=='Journal']['Debit_Amount'].sum()
    net_profit  = gross_margin - operating_exp
    cash_in     = d[d['Txn_Category']=='Receipt']['Credit_Amount'].sum()
    cash_out    = d[d['Txn_Category'].isin(['Payment','Journal'])]['Debit_Amount'].sum()
    net_cf      = cash_in - cash_out
    metrics[fy] = {
        'Net Sales': net_sales,
        'Purchase': purchase,
        'Gross Margin': gross_margin,
        'Net Profit': net_profit,
        'Cash In': cash_in,
        'Net Cash Flow': net_cf
    }

print(f'{"Metric":<20} {"FY 24-25":>12} {"FY 25-26":>12} {"Growth":>10}')
print('-' * 58)
for metric in metrics['2024-25']:
    v2425 = metrics['2024-25'][metric]
    v2526 = metrics['2025-26'][metric]
    growth = ((v2526 - v2425) / abs(v2425) * 100) if v2425 != 0 else 0
    arrow = '📈' if growth > 0 else '📉'
    print(f'{metric:<20} {v2425/10000000:>10.2f}Cr {v2526/10000000:>10.2f}Cr {arrow}{growth:>+6.1f}%')

print()
print('✅ Data cleaning complete! Load day_book_final.csv into Power BI.')

## Step 10 — DAX Measures Reference

Use these DAX measures in Power BI after loading `day_book_final.csv`:

```dax
-- SALES
Total Sales = 
    CALCULATE(SUM(day_book_final[Debit_Amount]), day_book_final[Txn_Category] = "Sales")
    + CALCULATE(SUM(day_book_final[Debit_Amount]), day_book_final[Txn_Category] = "Debit Note")
    - CALCULATE(SUM(day_book_final[Credit_Amount]), day_book_final[Txn_Category] = "Credit Note")

Total Sales Cr = DIVIDE([Total Sales], 10000000)

-- PURCHASE
Total Purchase = CALCULATE(SUM(day_book_final[Credit_Amount]), day_book_final[Txn_Category] = "Purchase")
Total Purchase Cr = DIVIDE([Total Purchase], 10000000)

-- GROSS MARGIN
Gross Margin = [Total Sales] - [Total Purchase]
Gross Margin Cr = DIVIDE([Gross Margin], 10000000)
Gross Margin % = DIVIDE([Gross Margin], [Total Sales]) * 100

-- OPERATING EXPENSES
Total Operating Expenses = CALCULATE(SUM(day_book_final[Debit_Amount]), day_book_final[Txn_Category] = "Journal")
Total Operating Expenses Cr = DIVIDE([Total Operating Expenses], 10000000)

-- NET PROFIT
Net Profit = [Gross Margin] - [Total Operating Expenses]
Net Profit Cr = DIVIDE([Net Profit], 10000000)
Net Profit % = DIVIDE([Net Profit], [Total Sales]) * 100

-- CASH FLOW
Total Cash In = CALCULATE(SUM(day_book_final[Credit_Amount]), day_book_final[Txn_Category] = "Receipt")
Total Cash Out = CALCULATE(SUM(day_book_final[Debit_Amount]), day_book_final[Txn_Category] IN {"Payment", "Journal"})
Net Cash Flow = [Total Cash In] - [Total Cash Out]
Net Cash Flow Cr = DIVIDE([Net Cash Flow], 10000000)

-- OWNER DRAWINGS
Owner Drawings = CALCULATE(SUM(day_book_final[Debit_Amount]), SEARCH("capital", day_book_final[Particulars], 1, 0) > 0)
Owner Drawings Cr = DIVIDE([Owner Drawings], 10000000)

-- LOAN PAYMENTS
Total Loan Payments = CALCULATE(SUM(day_book_final[Debit_Amount]), SEARCH("loan", day_book_final[Particulars], 1, 0) > 0)
Total Loan Payments Cr = DIVIDE([Total Loan Payments], 10000000)

-- CUSTOMERS
Num Customers = DISTINCTCOUNT(day_book_final[Particulars])
```